# Sacred Speech Scenario Planner — v3

**Version 3.0** · Distributed via GitHub · Each user supplies their own OpenAI API key.

Rapidly plan context-appropriate sacred speech (devotionals, homilies, memorial remarks, trauma responses) for any Army scenario. Combines a transparent **rules-based profiler** with an **LLM-assisted draft** that treats the chaplain's own scripture, illustrations, and concept as authoritative source material.

## Open in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KirtisChristensen/sacred-speech-planner/blob/main/Sacred_Speech_Planner_v3.ipynb)

> **Save your own copy.** In Colab: **File → Save a copy in Drive**. Edit your copy, not the canonical version. Pulling updates from GitHub is one click; your edits live in your Drive copy.

## How to use

> **First-time setup (each user, once):** the notebook author's API key is **not** shared via this link. Bring your own.
>
> 1. Get an OpenAI key at <https://platform.openai.com/api-keys> (charged to your billing — typical full run ≈ \$0.005 on `gpt-4o-mini`).
> 2. In Colab, click the **key icon** in the left sidebar → **+ Add new secret** → name it `OPENAI_API_KEY`, paste the value, toggle **Notebook access ON**.
> 3. Run **Step 0.5**. The first time, click **Grant access** on the consent popup. It should print `API_KEY loaded: length=…`.
> 4. Run **Step 0.6** (key health check) to confirm the key actually works before investing time in a scenario.

Then run the steps in order (`Shift+Enter`):

1. **Step 0.5** — confirm your API key loaded.
2. **Step 0.6** — sanity-check the key with a 1-token call.
3. **Step 1** — paste a natural-language scenario description → click **Autofill scenario**.
4. **Step 2** — review and edit the form fields the LLM populated.
5. **Step 3** — click **Generate Plan** for the rules-based profile and organizer.
6. **Step 4** — export the plan to Markdown (optional).
7. **Step 5** — enter your concept, scripture, illustrations, or draft (optional but recommended).
8. **Step 6** — click **Generate AI Draft**. Uses Step 5 content as authoritative, shaped by Step 3 profile and guardrails.

## OPSEC reminder

Use **generic descriptors only** — no PII, names, unit identifiers, or sensitive operational detail. Inputs in Steps 1 and 5 are sent to the OpenAI API.

## Pluralism

The tool is descriptive and culturally adaptive. It does not prescribe theology. The chaplain supplies all sacred text and doctrinal content. See `docs/PLURALISM.md` in the repo.

In [ ]:
# Step 0: Imports
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from dataclasses import dataclass, field, asdict
from typing import List
import textwrap

print('Imports OK. ipywidgets version:', widgets.__version__)

## Step 0.5 — LLM Configuration

Configure the OpenAI-compatible client used by autofill (Step 1.5) and AI draft (Step 4).

**Colab setup:** Click the **key icon** (left sidebar) → add a secret named `OPENAI_API_KEY` → toggle **Notebook access ON**.

Optional secrets: `LLM_PROVIDER` (`openai` or `xai`), `LLM_MODEL` (e.g. `gpt-4o-mini`), `XAI_API_KEY`.

**Local Jupyter:** set the same names as environment variables before launching.

**OPSEC reminder:** do not enter PII, names, unit IDs, or sensitive operational detail. Use generic descriptors.

In [ ]:
# Step 0.5: LLM configuration (OpenAI-compatible client)

import os

# Pull Colab Secrets into env vars (no-op if not on Colab)
try:
    from google.colab import userdata  # type: ignore
    for _k in ('OPENAI_API_KEY', 'XAI_API_KEY', 'LLM_PROVIDER', 'LLM_MODEL'):
        try:
            _v = userdata.get(_k)
            if _v and not os.environ.get(_k):
                os.environ[_k] = _v
        except Exception:
            pass
except Exception:
    pass

try:
    from openai import OpenAI
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openai'])
    from openai import OpenAI

PROVIDER = os.environ.get('LLM_PROVIDER', 'openai').lower()

if PROVIDER == 'openai':
    API_KEY  = os.environ.get('OPENAI_API_KEY', 'YOUR_KEY_HERE')
    BASE_URL = 'https://api.openai.com/v1'
    MODEL    = os.environ.get('LLM_MODEL', 'gpt-4o-mini')
elif PROVIDER == 'xai':
    API_KEY  = os.environ.get('XAI_API_KEY', 'YOUR_KEY_HERE')
    BASE_URL = 'https://api.x.ai/v1'
    MODEL    = os.environ.get('LLM_MODEL', 'grok-4')
else:
    raise ValueError("Unsupported PROVIDER. Use 'openai' or 'xai'.")

# ── DIRECT-KEY OVERRIDE (uncomment + paste for quickest path) ─────────────────
# API_KEY = "sk-PASTE-YOUR-OPENAI-KEY-HERE"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

print(f'Provider: {PROVIDER}')
print(f'LLM configured: {MODEL} @ {BASE_URL}')
print(f'API_KEY loaded: length={len(API_KEY) if API_KEY else 0}, starts with {API_KEY[:4]!r}')
if API_KEY in (None, '', 'YOUR_KEY_HERE'):
    print('\n⚠️  No API key detected. Add OPENAI_API_KEY to Colab Secrets, or uncomment the direct-key line above.')

def llm_chat(system: str, user: str, temperature: float = 0.4, response_format=None) -> str:
    """Shared helper for all LLM calls in this notebook."""
    kwargs = dict(
        model=MODEL,
        messages=[{'role': 'system', 'content': system},
                  {'role': 'user', 'content': user}],
        temperature=temperature,
    )
    if response_format is not None:
        kwargs['response_format'] = response_format
    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content

## Step 0.6 — Key health check

Quick sanity-check that your API key actually works before you invest time in a scenario. Sends a tiny prompt and prints the response.

In [ ]:
# Step 0.6: Key health check (1-token round trip)

if API_KEY in (None, '', 'YOUR_KEY_HERE'):
    print('❌  No API key. Go back to Step 0.5 and follow the first-time setup at the top of the notebook.')
else:
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': 'Reply with the single word: ready.'},
                {'role': 'user', 'content': 'ping'},
            ],
            max_tokens=5,
            temperature=0,
        )
        reply = (resp.choices[0].message.content or '').strip()
        print(f'✅  Key works. Model {MODEL} replied: {reply!r}')
    except Exception as e:
        print(f'❌  Test call failed: {e}')
        print('Common causes: invalid key, no billing on the account, wrong PROVIDER, or model name not available to your key.')

## Step 1 — Scenario Description (natural language)

Type or paste a free-form description of the event, audience, time, environment, and any constraints. Click **Autofill scenario** and the LLM will populate every form field for **Event**, **Audience**, **Time & Goals**, and **Chaplain notes**.

You will then **review and edit** those fields in Step 2 before profiling.

**OPSEC reminder:** generic descriptors only — no PII, names, unit IDs, or sensitive operational detail.

Example input:

> "Field memorial tomorrow morning for a soldier lost in a training accident. About 60 troops from the same company plus a few family members will attend. They've been working long shifts and are exhausted and grieving. I have about 10 minutes. Mixed faith backgrounds. Need to acknowledge the loss without rushing to platitudes."

In [ ]:
# Step 1: Build form widgets (silently) + Scenario Description input

import json

STYLE = {'description_width': '180px'}
LAYOUT = widgets.Layout(width='620px')

# ── Form widgets (created here, displayed in Step 2) ─────────────────────────
w_event_type = widgets.Dropdown(
    options=[
        'Field devotional',
        'Memorial service',
        'Graveside remarks',
        'Post-trauma debrief / response',
        'Disaster recovery worship',
        'Deployment send-off',
        'Redeployment / homecoming',
        'Daily "word of the day"',
        'Unit resilience / training event',
        'Garrison chapel service',
        'Other',
    ],
    value='Disaster recovery worship',
    description='Event type:', style=STYLE, layout=LAYOUT,
)
w_event_other = widgets.Text(description='If Other, specify:', style=STYLE, layout=LAYOUT)

w_audience_primary = widgets.Dropdown(
    options=[
        'Soldiers in active recovery / response',
        'Exhausted work crews',
        'Bereaved family / next of kin',
        'Mixed unit formation',
        'Leadership / command team',
        'Families and dependents',
        'Trainees / initial entry',
        'Mixed civilian + military',
    ],
    value='Exhausted work crews',
    description='Primary audience:', style=STYLE, layout=LAYOUT,
)
w_audience_size = widgets.Dropdown(
    options=['Small (<20)', 'Medium (20-100)', 'Large formation (100+)'],
    value='Small (<20)',
    description='Audience size:', style=STYLE, layout=LAYOUT,
)
w_trauma = widgets.Dropdown(
    options=['Low', 'Medium', 'High', 'Acute / immediate'],
    value='High',
    description='Trauma level:', style=STYLE, layout=LAYOUT,
)
w_fatigue = widgets.Dropdown(
    options=['Low', 'Medium', 'High', 'Severe (sustained ops)'],
    value='High',
    description='Fatigue level:', style=STYLE, layout=LAYOUT,
)
w_orientation = widgets.Dropdown(
    options=['Mostly emotional', 'Balanced', 'Mostly cognitive'],
    value='Mostly emotional',
    description='Audience orientation:', style=STYLE, layout=LAYOUT,
)
w_cultural_mix = widgets.Dropdown(
    options=['Homogeneous', 'Moderately mixed', 'Highly pluralistic'],
    value='Highly pluralistic',
    description='Cultural / faith mix:', style=STYLE, layout=LAYOUT,
)
w_time = widgets.Dropdown(
    options=['3-5 min', '5-10 min', '10-15 min', '15-20 min', '20+ min'],
    value='5-10 min',
    description='Available time:', style=STYLE, layout=LAYOUT,
)
w_optempo = widgets.Dropdown(
    options=['Immediate response', 'Sustained operations', 'Recovery phase', 'Garrison / steady state'],
    value='Immediate response',
    description='OPTEMPO:', style=STYLE, layout=LAYOUT,
)
w_location = widgets.Dropdown(
    options=['Field / outdoors', 'Motor pool', 'Chapel tent', 'Disaster site', 'Briefing room', 'Garrison chapel', 'Other'],
    value='Disaster site',
    description='Location:', style=STYLE, layout=LAYOUT,
)
w_situation = widgets.Dropdown(
    options=['Natural disaster', 'Combat loss', 'Training accident', 'Routine resilience', 'Deployment cycle', 'Suicide / line-of-duty death', 'Other'],
    value='Natural disaster',
    description='Broader situation:', style=STYLE, layout=LAYOUT,
)
w_needs = widgets.SelectMultiple(
    options=['Comfort', 'Encouragement', 'Hope', 'Lament / processing grief', 'Meaning-making', 'Resilience', 'Communal bonding', 'Individual reflection', 'Challenge / call to action'],
    value=('Comfort', 'Hope', 'Lament / processing grief'),
    description='Needs to address:', style=STYLE, layout=widgets.Layout(width='620px', height='150px'),
)
w_tradition = widgets.Text(
    value='',
    placeholder='(optional, internal reference only)',
    description='Chaplain tradition:', style=STYLE, layout=LAYOUT,
)
w_text_pref = widgets.Text(
    value='',
    placeholder='e.g., Psalm 46, parable preference, etc. (optional)',
    description='Sacred text pref:', style=STYLE, layout=LAYOUT,
)

form = widgets.VBox([
    widgets.HTML('<h3>Event</h3>'),
    w_event_type, w_event_other, w_situation, w_location, w_optempo,
    widgets.HTML('<h3>Audience</h3>'),
    w_audience_primary, w_audience_size, w_trauma, w_fatigue, w_orientation, w_cultural_mix,
    widgets.HTML('<h3>Time & Goals</h3>'),
    w_time, w_needs,
    widgets.HTML('<h3>Chaplain notes (optional)</h3>'),
    w_tradition, w_text_pref,
])

# ── Autofill: map field name -> (widget, allowed options or None) ────────────
_AUTOFILL_FIELDS = {
    'event_type':       (w_event_type,       list(w_event_type.options)),
    'event_other':      (w_event_other,      None),
    'situation':        (w_situation,        list(w_situation.options)),
    'location':         (w_location,         list(w_location.options)),
    'optempo':          (w_optempo,          list(w_optempo.options)),
    'audience_primary': (w_audience_primary, list(w_audience_primary.options)),
    'audience_size':    (w_audience_size,    list(w_audience_size.options)),
    'trauma':           (w_trauma,           list(w_trauma.options)),
    'fatigue':          (w_fatigue,          list(w_fatigue.options)),
    'orientation':      (w_orientation,      list(w_orientation.options)),
    'cultural_mix':     (w_cultural_mix,     list(w_cultural_mix.options)),
    'time':             (w_time,             list(w_time.options)),
    'text_pref':        (w_text_pref,        None),
    'tradition':        (w_tradition,        None),
}
_NEEDS_ALLOWED = list(w_needs.options)

AUTOFILL_SYSTEM = """You are an assistant helping a U.S. Army chaplain plan sacred speech.
Read the chaplain's free-form scenario description and produce a JSON object that
maps onto a structured form. Pick the BEST option from each allowed list.
If a field is genuinely unspecified, use the provided default.
Do NOT invent PII or unit identifiers. Output JSON only — no prose, no markdown.

Required JSON keys and allowed values:

- event_type: one of ["Field devotional","Memorial service","Graveside remarks",
  "Post-trauma debrief / response","Disaster recovery worship","Deployment send-off",
  "Redeployment / homecoming","Daily \\"word of the day\\"",
  "Unit resilience / training event","Garrison chapel service","Other"]
- event_other: free text (only if event_type == "Other"; else "")
- situation: one of ["Natural disaster","Combat loss","Training accident",
  "Routine resilience","Deployment cycle","Suicide / line-of-duty death","Other"]
- location: one of ["Field / outdoors","Motor pool","Chapel tent","Disaster site",
  "Briefing room","Garrison chapel","Other"]
- optempo: one of ["Immediate response","Sustained operations","Recovery phase",
  "Garrison / steady state"]
- audience_primary: one of ["Soldiers in active recovery / response",
  "Exhausted work crews","Bereaved family / next of kin","Mixed unit formation",
  "Leadership / command team","Families and dependents","Trainees / initial entry",
  "Mixed civilian + military"]
- audience_size: one of ["Small (<20)","Medium (20-100)","Large formation (100+)"]
- trauma: one of ["Low","Medium","High","Acute / immediate"]
- fatigue: one of ["Low","Medium","High","Severe (sustained ops)"]
- orientation: one of ["Mostly emotional","Balanced","Mostly cognitive"]
- cultural_mix: one of ["Homogeneous","Moderately mixed","Highly pluralistic"]
- time: one of ["3-5 min","5-10 min","10-15 min","15-20 min","20+ min"]
- needs: array of zero or more from ["Comfort","Encouragement","Hope",
  "Lament / processing grief","Meaning-making","Resilience","Communal bonding",
  "Individual reflection","Challenge / call to action"]
- text_pref: free text scripture/text preference if explicitly mentioned, else ""
- tradition: free text chaplain tradition if explicitly mentioned, else ""
- rationale: 1-3 short bullets explaining your key choices (string with line breaks).

Defaults if truly unspecified: event_type="Field devotional", situation="Other",
location="Field / outdoors", optempo="Garrison / steady state",
audience_primary="Mixed unit formation", audience_size="Small (<20)",
trauma="Low", fatigue="Low", orientation="Balanced", cultural_mix="Highly pluralistic",
time="5-10 min", needs=["Encouragement","Hope"], text_pref="", tradition="".
"""

# ── Scenario description input + Autofill button ─────────────────────────────
w_nl_desc = widgets.Textarea(
    value='',
    placeholder='Describe the event, audience, time available, environment, '
                'fatigue/trauma indicators, and any constraints. '
                'Generic descriptors only — no PII.',
    description='Scenario:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='820px', height='200px'),
)
w_autofill_btn = widgets.Button(description='Autofill scenario', button_style='info', icon='magic')
w_autofill_out = widgets.Output()

def _apply_autofill(data: dict):
    for key, (widget, allowed) in _AUTOFILL_FIELDS.items():
        if key not in data:
            continue
        val = data[key]
        if allowed is not None and val not in allowed:
            print(f'  • {key}: ignored "{val}" (not in allowed list)')
            continue
        widget.value = val
    needs = data.get('needs', []) or []
    needs = [n for n in needs if n in _NEEDS_ALLOWED]
    if needs:
        w_needs.value = tuple(needs)

def on_autofill_click(_):
    with w_autofill_out:
        clear_output()
        if API_KEY in (None, '', 'YOUR_KEY_HERE'):
            print('No API key. Configure Step 0.5 first.')
            return
        desc = w_nl_desc.value.strip()
        if not desc:
            print('Type a scenario description first.')
            return
        print(f'Calling {MODEL} for scenario extraction…')
        try:
            raw = llm_chat(
                AUTOFILL_SYSTEM,
                desc,
                temperature=0.1,
                response_format={'type': 'json_object'},
            )
        except Exception as e:
            print(f'LLM call failed: {e}')
            return
        try:
            data = json.loads(raw)
        except Exception as e:
            print(f'Could not parse JSON: {e}')
            print('Raw output:')
            print(raw)
            return
        clear_output()
        print('Autofill applied. Now go to Step 2 to review and edit the form fields.\n')
        _apply_autofill(data)
        if data.get('rationale'):
            display(Markdown('**LLM rationale:**\n\n' + str(data['rationale'])))
        print('\nFinal extracted JSON:')
        print(json.dumps(data, indent=2))

w_autofill_btn.on_click(on_autofill_click)
display(widgets.VBox([w_nl_desc, w_autofill_btn, w_autofill_out]))

## Step 2 — Review and Edit Scenario Inputs

The form below shows the values from autofill (or the defaults if you skipped Step 1). **Review every field** and edit anything the LLM got wrong, especially **Time**, **Trauma level**, **Fatigue level**, and **Cultural / faith mix** — these drive the recommended structure and tone.

In [ ]:
# Step 2: Display the form for review and edits
display(form)

## Step 3 — Generate Scenario Profile

The profiler maps your inputs to a recommended **form**, **tone**, **structure**, and **trauma-responsiveness posture** using descriptive rules drawn from SMFF, cultural intelligence, and trauma-responsive principles. It does **not** prescribe theology.

In [ ]:
# Step 2: Profiling logic (rules-based)

@dataclass
class ScenarioInputs:
    event_type: str
    event_other: str
    situation: str
    location: str
    optempo: str
    audience_primary: str
    audience_size: str
    trauma: str
    fatigue: str
    orientation: str
    cultural_mix: str
    time: str
    needs: List[str]
    tradition: str
    text_pref: str

@dataclass
class ScenarioProfile:
    summary: str
    recommended_form: str
    target_length: str
    structure: List[str]   # ordered (section, time, intent)
    tone: List[str]
    trauma_posture: List[str]
    cultural_posture: List[str]
    delivery_tips: List[str]
    avoid: List[str]

def collect_inputs() -> ScenarioInputs:
    return ScenarioInputs(
        event_type=w_event_type.value,
        event_other=w_event_other.value.strip(),
        situation=w_situation.value,
        location=w_location.value,
        optempo=w_optempo.value,
        audience_primary=w_audience_primary.value,
        audience_size=w_audience_size.value,
        trauma=w_trauma.value,
        fatigue=w_fatigue.value,
        orientation=w_orientation.value,
        cultural_mix=w_cultural_mix.value,
        time=w_time.value,
        needs=list(w_needs.value),
        tradition=w_tradition.value.strip(),
        text_pref=w_text_pref.value.strip(),
    )

def _is_high(level: str) -> bool:
    return level in ('High', 'Acute / immediate', 'Severe (sustained ops)')

def _time_minutes(t: str) -> int:
    # Use the lower bound for planning
    return {'3-5 min': 3, '5-10 min': 5, '10-15 min': 10, '15-20 min': 15, '20+ min': 20}.get(t, 7)

def profile(inp: ScenarioInputs) -> ScenarioProfile:
    minutes = _time_minutes(inp.time)
    high_trauma = _is_high(inp.trauma)
    high_fatigue = _is_high(inp.fatigue)
    pluralistic = inp.cultural_mix in ('Moderately mixed', 'Highly pluralistic')
    emotional = inp.orientation == 'Mostly emotional'

    # --- Recommended form ---
    et = inp.event_type
    if et == 'Memorial service':
        form = 'Memorial homily — Lament + Hope arc'
    elif et == 'Graveside remarks':
        form = 'Graveside remarks — brief, dignified, pastoral'
    elif et == 'Post-trauma debrief / response':
        form = 'Trauma-responsive presence message (Acknowledge–Pivot–Hope)'
    elif et == 'Disaster recovery worship':
        form = 'Short field devotional — trauma-responsive hope message'
    elif et == 'Deployment send-off':
        form = 'Send-off charge — courage + connection + blessing'
    elif et == 'Redeployment / homecoming':
        form = 'Homecoming reflection — gratitude + reintegration'
    elif et == 'Daily "word of the day"':
        form = 'Micro-devotional — single image, single point'
    elif et == 'Unit resilience / training event':
        form = 'Resilience teaching — story + principle + application'
    elif et == 'Garrison chapel service':
        form = 'Standard homily — narrative arc with one clear point'
    else:
        form = 'Adaptive short-form message (Acknowledge–Pivot–Hope)'

    if high_trauma and minutes <= 10 and 'devotional' not in form.lower() and 'graveside' not in form.lower():
        form += ' [trauma-responsive, ultra-concise variant]'

    # --- Structure with timing ---
    structure: List[str] = []
    if high_trauma or et in ('Post-trauma debrief / response', 'Disaster recovery worship', 'Memorial service', 'Graveside remarks'):
        # Acknowledge-Pivot-Hope core
        a = max(1, round(minutes * 0.25))
        c = max(1, round(minutes * 0.45))
        h = max(1, minutes - a - c)
        structure = [
            f'1. Acknowledge (~{a} min) — name the loss/exhaustion/reality without minimizing.',
            f'2. Anchor (~{c} min) — one short sacred-text or story anchor; one clear idea.',
            f'3. Hope / Blessing (~{h} min) — transcendent pivot, presence, and closing blessing.',
        ]
    elif et == 'Daily "word of the day"':
        structure = [
            '1. One image or phrase (~30 sec).',
            '2. One sentence of meaning (~1 min).',
            '3. One sending line (~30 sec).',
        ]
    else:
        # Narrative arc
        i = max(1, round(minutes * 0.15))
        b = max(1, round(minutes * 0.55))
        c = max(1, minutes - i - b)
        structure = [
            f'1. Hook / context (~{i} min) — meet the audience where they are.',
            f'2. Body (~{b} min) — one main point, one story, one sacred-text anchor.',
            f'3. Call / blessing (~{c} min) — clear application or sending.',
        ]

    # --- Tone ---
    tone = []
    if high_trauma:
        tone += ['Empathetic', 'Lament + hope', 'Calm, slow, grounded']
    else:
        tone += ['Hope-focused', 'Warm and direct']
    if emotional:
        tone.append('Heart-forward (image and story over argument)')
    elif inp.orientation == 'Mostly cognitive':
        tone.append('Clear, structured, one explicit point')
    else:
        tone.append('Balanced head/heart')
    if high_fatigue:
        tone.append('Low cognitive load — short sentences, repeated key phrase')

    # --- Trauma posture ---
    trauma_posture = []
    if high_trauma:
        trauma_posture = [
            'Validate suffering before pivoting to hope.',
            'Use lament language; avoid platitudes ("everything happens for a reason", "be strong").',
            'Emphasize presence ("you are not alone") over explanation.',
            'No graphic re-description of the event.',
            'Offer a concrete next-step image (small, doable).',
        ]
    else:
        trauma_posture = [
            'Stay audience-centered; assume some carry hidden weight.',
            'Avoid platitudes; keep hope grounded in real action or presence.',
        ]

    # --- Cultural posture ---
    cultural_posture = []
    if pluralistic:
        cultural_posture = [
            'Speak from your tradition without requiring shared belief.',
            'Use universal human anchors (breath, light, presence, courage) alongside sacred text.',
            'Avoid in-group shorthand or assumed doctrinal terms.',
            'Offer blessing in inclusive form; let listeners receive it from their own tradition.',
        ]
    else:
        cultural_posture = [
            'You may use tradition-specific language with care; still honor any outliers present.',
        ]

    # --- Delivery tips ---
    delivery_tips = [
        'Short sentences. Plain language. No jargon.',
        'Slow pace; deliberate pauses after key lines.',
        'Eye contact across the formation, not just the front row.',
        f'Hard time cap: {minutes} min. Cut content, not pace.',
    ]
    if inp.location in ('Field / outdoors', 'Motor pool', 'Disaster site'):
        delivery_tips.append('Project voice; assume wind/equipment noise; no notes-shuffling.')
    if inp.audience_size == 'Large formation (100+)':
        delivery_tips.append('Larger gestures; one clear repeated phrase to anchor the back rows.')
    if high_fatigue:
        delivery_tips.append('Open with permission to sit/rest; do not require participation.')

    # --- Avoid list ---
    avoid = []
    if high_trauma:
        avoid += [
            'Triumphalism or quick resolution of grief.',
            'Detailed re-narration of the traumatic event.',
            'Calls to action that demand emotional output.',
        ]
    if pluralistic:
        avoid.append('Insider phrases that require shared doctrine to land.')
    if minutes <= 10:
        avoid.append('Multi-point sermons; pick ONE idea.')
    if high_fatigue:
        avoid.append('Long preambles, throat-clearing, or thanking lists.')

    # --- Summary line ---
    descriptors = []
    if high_trauma: descriptors.append('high-trauma')
    if high_fatigue: descriptors.append('high-fatigue')
    descriptors.append(f'{minutes}-min')
    descriptors.append(inp.optempo.lower())
    summary = (
        f"This is a **{', '.join(descriptors)}** event — "
        f"{inp.audience_primary.lower()} at a {inp.location.lower()} "
        f"during {inp.situation.lower()}. "
        f"Cultural mix: {inp.cultural_mix.lower()}. "
        f"Audience need(s): {', '.join(inp.needs) if inp.needs else '(unspecified)'}."
    )

    return ScenarioProfile(
        summary=summary,
        recommended_form=form,
        target_length=inp.time,
        structure=structure,
        tone=tone,
        trauma_posture=trauma_posture,
        cultural_posture=cultural_posture,
        delivery_tips=delivery_tips,
        avoid=avoid,
    )

print('Profiler defined. Run the next cell to generate a plan.')

In [ ]:
# Step 2b: Generate button + render

out = widgets.Output()
btn = widgets.Button(description='Generate Plan', button_style='primary', icon='check')

def render_profile(p: ScenarioProfile, inp: ScenarioInputs) -> str:
    def bullets(items):
        return '\n'.join(f'- {x}' for x in items) if items else '- (none)'
    text_anchor = inp.text_pref if inp.text_pref else '_(chaplain selects)_'
    tradition = inp.tradition if inp.tradition else '_(not specified)_'
    md = f"""
## A. Scenario Profile Summary
{p.summary}

## B. Recommended Sacred Speech Form
**{p.recommended_form}**  
Target length: **{p.target_length}**

### Structure & Timing
{bullets(p.structure)}

### Tone
{bullets(p.tone)}

### Trauma-Responsive Posture
{bullets(p.trauma_posture)}

### Cultural / Pluralism Posture
{bullets(p.cultural_posture)}

### Avoid
{bullets(p.avoid)}

## C. Sacred Speech Organizer (fillable)
_Chaplain supplies all theological content. The structure below mirrors the recommended form._

```
Title / Theme:

Scripture / Sacred Text Anchor: {text_anchor}

Context Acknowledgment (validate trauma/fatigue/time):

Core Message (ONE clear point):

Illustration / Story (short, military-relevant, non-graphic):

Hope / Transcendent Pivot:

Call to Action or Closing Blessing:

Chaplain tradition note (internal): {tradition}
```

## D. Delivery & Preparation Tips
{bullets(p.delivery_tips)}

**Preparation shortcut (under OPTEMPO):**
1. Write the ONE sentence of the core message first.
2. Pick ONE sacred-text anchor and ONE short illustration.
3. Draft Acknowledge → Anchor → Hope in that order. Read it aloud once with a timer.
4. Cut anything that does not serve the one sentence.

## E. Feedback & Growth Plan (post-event)
- **SMFF self-score (1–5)** on: Clarity, Relevance, Empathy, Time discipline, Pluralism fit.
- **Audience signal**: 1–2 trusted listeners give honest reception feedback.
- **IDP next step**: Practice this same form in a low-stakes setting within 30 days.
- **Track** scores across rating period to show measurable growth.
"""
    return md

_last_profile = {'inp': None, 'profile': None, 'markdown': None}

def on_click(_):
    with out:
        clear_output()
        inp = collect_inputs()
        p = profile(inp)
        md = render_profile(p, inp)
        _last_profile['inp'] = inp
        _last_profile['profile'] = p
        _last_profile['markdown'] = md
        display(Markdown(md))

btn.on_click(on_click)
display(btn, out)

## Step 4 — Save / Export Plan (optional)

Run the next cell to save the rules-based plan as a Markdown file alongside the notebook. (On Colab, use the Files panel to download, or mount Drive separately.) Stop here if you only need the rules-based outline.

In [ ]:
# Step 3: Export last generated plan to Markdown
import datetime, os

if _last_profile['markdown'] is None:
    print('No plan generated yet. Run Step 2 and click Generate Plan first.')
else:
    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    fname = f'sacred_speech_plan_{stamp}.md'
    with open(fname, 'w', encoding='utf-8') as f:
        f.write(f'# Sacred Speech Plan ({stamp})\n')
        f.write(_last_profile['markdown'])
    print(f'Saved: {os.path.abspath(fname)}')

## Step 5 — Chaplain Content (optional, but recommended)

If you stop here, the AI draft in Step 6 will be generated **only** from the rules-based profile — generic by design.

To produce a draft that reflects **your** message, fill in any of the fields below. Anything you provide becomes the **authoritative source** the LLM must build on (it will not override your scripture choices, illustrations, or core message).

You may provide as little as a one-line concept, or as much as a full draft you want polished/tightened.

In [ ]:
# Step 3.5: Chaplain content inputs

w_chap_concept = widgets.Textarea(
    value='',
    placeholder='One or two sentences: what do you want the audience to walk away with?',
    description='Core concept:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='820px', height='80px'),
)

w_chap_scripture = widgets.Textarea(
    value='',
    placeholder='e.g., Psalm 46:1-3 (full text or reference); Lamentations 3:22-23; story of Elijah at Horeb. '
                'Include exact text if you want it quoted; references-only is fine if you trust the model less.',
    description='Scripture / texts:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='820px', height='120px'),
)

w_chap_illustrations = widgets.Textarea(
    value='',
    placeholder='Short stories, analogies, military examples you want included. Keep them non-graphic.',
    description='Illustrations:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='820px', height='120px'),
)

w_chap_draft = widgets.Textarea(
    value='',
    placeholder='Optional: existing draft text or notes to be tightened, time-fitted, and shaped into the recommended structure. '
                'The LLM will preserve your wording where possible.',
    description='Draft / notes:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='820px', height='200px'),
)

w_chap_constraints = widgets.Textarea(
    value='',
    placeholder='Optional: things to include or avoid (e.g., "do not quote 1 Cor 13", "must mention unit motto in inclusive way").',
    description='Other constraints:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='820px', height='80px'),
)

display(widgets.VBox([
    widgets.HTML('<h3>Your message and source material</h3>'
                 '<p style="color:#666;font-size:90%">All fields optional. Provide what you have. '
                 'The LLM treats your inputs as authoritative — it will tighten and shape, not overrule.</p>'),
    w_chap_concept,
    w_chap_scripture,
    w_chap_illustrations,
    w_chap_draft,
    w_chap_constraints,
]))

def collect_chaplain_content() -> dict:
    return {
        'concept':       w_chap_concept.value.strip(),
        'scripture':     w_chap_scripture.value.strip(),
        'illustrations': w_chap_illustrations.value.strip(),
        'draft':         w_chap_draft.value.strip(),
        'constraints':   w_chap_constraints.value.strip(),
    }

## Step 6 — Generate AI Draft

Combines three sources, in this priority order:

1. **Chaplain content** from Step 5 (authoritative — preserved).
2. **Step 3 profile** — recommended form, structure, tone, trauma/pluralism posture (shaping).
3. **Hard guardrails** in the system prompt — pluralism-safe, trauma-responsive, time-capped, no platitudes, no PII echo, draft-only.

LLM was configured in Step 0.5. If you skipped Step 5, the draft will be generic; provide content there for a draft that reflects your message.

In [ ]:
# Step 4: Prompt builder + Generate AI Draft button (v2 — uses chaplain content)

DRAFT_SYSTEM_PROMPT = """You are an assistant helping a U.S. Army chaplain shape SACRED SPEECH (devotionals, homilies, memorial remarks, blessings) for a specific scenario.

You will receive THREE inputs:
A) A SCENARIO PROFILE (form, structure, tone, trauma/pluralism posture, time cap).
B) Optional CHAPLAIN CONTENT — concept, scripture/texts, illustrations, draft, constraints.
C) Hard rules below.

PRIORITY OF SOURCES (highest first):
1. Chaplain Content. Treat it as authoritative. Do NOT replace the chaplain's scripture choices, illustrations, core message, or wording with your own preferences. Tighten, fit to time, place into structure, polish — do not overrule.
2. Scenario Profile. Use it to shape structure, tone, length, and posture.
3. Your own additions. Only fill gaps the chaplain left blank, and clearly mark them as suggestions in the "Notes for the Chaplain" section.

HARD RULES — never violate:
1. Pluralism-safe. Inclusive language for a mixed military formation. Do NOT prescribe doctrine. If the chaplain provided tradition-specific text or wording, keep it; do not water it down — but frame it so listeners from other traditions can still receive the message with respect.
2. Trauma-responsive when indicated. Validate suffering before pivoting to hope. No platitudes ("everything happens for a reason," "be strong," "God needed another angel"). No graphic re-narration of traumatic events. Emphasize presence over explanation.
3. Time discipline. Stay within the chaplain's time cap (≈130 words/min spoken, then trim 20% for pauses). Prefer ONE clear core idea.
4. Plain language. Short sentences. No jargon or insider shorthand that requires shared belief.
5. Audience respect. Never demand emotional output from exhausted, grieving, or traumatized listeners.
6. No PII. Do not invent names, units, locations, or identifying detail. Use only generic descriptors provided.
7. Draft only. This is a starting draft for the chaplain to edit. Do not claim authority.
8. Scripture handling. If the chaplain provided full scripture text, you may quote it. If only a reference, paraphrase carefully and note "verify exact wording" in the chaplain notes — do NOT fabricate quotations.

OUTPUT FORMAT (Markdown), in this exact order, with these headings:

### Title / Theme (draft)
### Context Acknowledgment (draft)
### Core Message — one sentence (draft)
### Scripture / Sacred Text Anchor (draft)
### Illustration / Story (short, non-graphic) (draft)
### Hope / Transcendent Pivot (draft)
### Closing Blessing (draft)
### Notes for the Chaplain
- Bullets: which sections came from chaplain content vs. were filled in by you, what to verify (especially scripture quotations), where to insert tradition-specific content, what to cut if time runs short.
"""

def _fmt_chaplain_block(c: dict) -> str:
    if not any(c.values()):
        return "(none — generate a generic draft from the profile only; mark all sections as model-filled in the notes.)"
    parts = []
    if c['concept']:
        parts.append(f"CORE CONCEPT (chaplain):\n{c['concept']}")
    if c['scripture']:
        parts.append(f"SCRIPTURE / SACRED TEXTS (chaplain — authoritative):\n{c['scripture']}")
    if c['illustrations']:
        parts.append(f"ILLUSTRATIONS / STORIES (chaplain — preserve wording where possible):\n{c['illustrations']}")
    if c['draft']:
        parts.append(f"EXISTING DRAFT / NOTES (chaplain — tighten and shape, do not rewrite the voice):\n{c['draft']}")
    if c['constraints']:
        parts.append(f"OTHER CONSTRAINTS:\n{c['constraints']}")
    return "\n\n".join(parts)

def build_user_prompt(inp, p, chaplain: dict) -> str:
    needs = ', '.join(inp.needs) if inp.needs else '(unspecified)'
    text_pref = inp.text_pref if inp.text_pref else '(see chaplain content below)'
    tradition = inp.tradition if inp.tradition else '(not specified)'
    structure = '\n'.join(p.structure)
    tone = ', '.join(p.tone)
    trauma_posture = '\n'.join(f'- {x}' for x in p.trauma_posture)
    cultural_posture = '\n'.join(f'- {x}' for x in p.cultural_posture)
    avoid = '\n'.join(f'- {x}' for x in p.avoid) if p.avoid else '- (none specified)'
    chaplain_block = _fmt_chaplain_block(chaplain)
    return f"""A) SCENARIO PROFILE
-------------------
Event type: {inp.event_type}{(' / ' + inp.event_other) if inp.event_other else ''}
Broader situation: {inp.situation}
Location: {inp.location}
OPTEMPO: {inp.optempo}

Audience: {inp.audience_primary}
Size: {inp.audience_size}
Trauma level: {inp.trauma}
Fatigue level: {inp.fatigue}
Orientation: {inp.orientation}
Cultural / faith mix: {inp.cultural_mix}

Time cap: {inp.time}
Needs to address: {needs}

Sacred-text preference (form field): {text_pref}
Chaplain tradition (internal reference only): {tradition}

RECOMMENDED FORM
{p.recommended_form}

REQUIRED STRUCTURE & TIMING
{structure}

TONE
{tone}

TRAUMA-RESPONSIVE POSTURE
{trauma_posture}

CULTURAL / PLURALISM POSTURE
{cultural_posture}

AVOID
{avoid}

B) CHAPLAIN CONTENT (AUTHORITATIVE — preserve wording, scripture, illustrations)
--------------------------------------------------------------------------------
{chaplain_block}

TASK
----
Produce a sacred-speech DRAFT following the priority order in your system instructions:
chaplain content first (preserved and shaped), profile second (for structure/tone/posture/time),
your additions only to fill gaps and clearly marked in the chaplain notes.
Honor every hard rule. Output only the Markdown sections specified.
"""

ai_out = widgets.Output()
ai_btn = widgets.Button(description='Generate AI Draft', button_style='success', icon='magic')

_last_ai = {'prompt': None, 'draft': None, 'chaplain': None}

def on_ai_click(_):
    with ai_out:
        clear_output()
        if _last_profile.get('profile') is None:
            print('Run Step 2 (Generate Plan) first so the profile is available.')
            return
        if API_KEY in (None, '', 'YOUR_KEY_HERE'):
            print('No API key configured. See Step 0.5.')
            return
        inp = _last_profile['inp']
        p = _last_profile['profile']
        chaplain = collect_chaplain_content()
        user_prompt = build_user_prompt(inp, p, chaplain)
        _last_ai['prompt'] = user_prompt
        _last_ai['chaplain'] = chaplain
        provided = [k for k, v in chaplain.items() if v]
        print(f'Calling {MODEL}…')
        if provided:
            print(f'Chaplain content provided: {", ".join(provided)}')
        else:
            print('No chaplain content provided — draft will be generic.')
        try:
            draft = llm_chat(DRAFT_SYSTEM_PROMPT, user_prompt, temperature=0.6)
        except Exception as e:
            print(f'LLM call failed: {e}')
            return
        _last_ai['draft'] = draft
        clear_output()
        display(Markdown('## F. AI Draft (chaplain edits before use)\n\n' + draft))
        display(Markdown('---\n*Draft only. Verify scripture quotations, pluralism fit, trauma posture, and time discipline before delivery.*'))

ai_btn.on_click(on_ai_click)
display(ai_btn, ai_out)

In [ ]:
# Step 4c: Export combined plan + chaplain content + AI draft to Markdown (optional)
import datetime, os

if _last_profile.get('markdown') is None:
    print('No plan generated yet. Run Step 2 first.')
elif _last_ai.get('draft') is None:
    print('No AI draft generated yet. Run Step 4 (Generate AI Draft) first.')
else:
    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    fname = f'sacred_speech_plan_with_draft_{stamp}.md'
    chaplain = _last_ai.get('chaplain') or {}
    with open(fname, 'w', encoding='utf-8') as f:
        f.write(f'# Sacred Speech Plan + AI Draft ({stamp})\n')
        f.write(_last_profile['markdown'])
        if any(chaplain.values()):
            f.write('\n\n## E2. Chaplain Content (source material)\n')
            for label, key in [('Core concept','concept'), ('Scripture / texts','scripture'),
                               ('Illustrations','illustrations'), ('Draft / notes','draft'),
                               ('Other constraints','constraints')]:
                if chaplain.get(key):
                    f.write(f'\n**{label}:**\n\n{chaplain[key]}\n')
        f.write('\n\n## F. AI Draft (chaplain edits before use)\n\n')
        f.write(_last_ai['draft'])
        f.write('\n\n---\n*Draft only. Verify scripture quotations, pluralism fit, trauma posture, and time discipline before delivery.*\n')
    print(f'Saved: {os.path.abspath(fname)}')

---

## Version

**Sacred Speech Scenario Planner v3.0** · MIT License · Source: <https://github.com/KirtisChristensen/sacred-speech-planner>

## Feedback & issues

- File issues at <https://github.com/KirtisChristensen/sacred-speech-planner/issues>.
- For each report please include: which step, what you input (generic, no PII), what the tool produced, and what you expected.

## Quick validation (run before relying on the tool)

1. **Autofill accuracy.** Paste a free-form scenario into Step 1. Confirm Step 2 form values are plausible. Edit any miss.
2. **Generic AI draft.** Skip Step 5; run Step 6. Confirm the draft is competent but generic and the "Notes for the Chaplain" section flags every part as model-filled.
3. **Chaplain-led AI draft.** Fill Step 5 with a one-line concept + one scripture passage + one short illustration. Confirm your scripture and wording are preserved, structure follows Step 3, time cap is respected.
4. **Pluralism stress test.** Highly pluralistic audience + tradition-specific scripture from chaplain. Tradition wording preserved; framing remains inclusive.
5. **Trauma posture stress test.** High-trauma scenario; no platitudes; presence emphasized over explanation.

## Roadmap (post-v3)

- Drive auto-export of plan + chaplain content + draft.
- Post-event SMFF scoring cell that records 1–5 ratings to a CSV for trend tracking.
- Prebuilt example scenarios as one-click presets (see `examples/` in the repo).
- Streamlit/Gradio port for a non-notebook UI (Phase 4).